# Test tile queue - C-band Luna OVA

Builds a LabExT experiment queue for the tiled test layout: six fibre array placements
("tiles"), each with one input grating coupler and four measurable devices read out on the
neighbouring channels.

**How the fibre array maps onto a tile**

The 7-channel fibre array is placed so that **channel 4 lands on the tile's Input grating
coupler**. The remaining channels then fall on the neighbouring couplers by position, so a
device's channel is simply

```
channel = 4 + (device GC index - input GC index)
```

The input GC is not always in the middle of a tile, so the set of channels in use differs
per tile (e.g. tile 1 uses 3/5/6/7, tile 4 uses 1/2/3/5). That is handled automatically
below - you only describe each tile in physical left-to-right order.

Same switch wiring as the other notebooks in this folder: N ports 1-7 on fibre array
channels 1-7, channel 4 is the input, Luna OVA on switch M ports 3 and 4.

In [1]:
# --- Hardware map -------------------------------------------------------------------
# Dicon GP800 switch: M ports are the instrument side, N ports the fibre-array side.
# N ports 1-7 are wired to fibre array channels 1-7.
FIBRE_ARRAY_CHANNELS = [1, 2, 3, 4, 5, 6, 7]

# Fibre array channel 4 launches light into the chip.
INPUT_CHANNEL = 4

# The Luna OVA occupies switch M ports 3 and 4: M3 carries the OVA source into the chip
# input, M4 returns the device output to the OVA. Swap these two if your OVA source is
# patched to M4 instead - nothing else in this notebook needs to change.
OVA_SOURCE_M_PORT = 3
OVA_RECEIVE_M_PORT = 4

# Measurement class name, as listed in the LabExT Experiment Wizard.
MEASUREMENT_CLASS = "LUNA_sweep_Cband_switch"

# Must match the chip currently imported in LabExT; device ids are looked up on it.
CHIP_NAME = "test_tile_markers"

In [2]:
# --- OVA sweep settings -------------------------------------------------------------
# Applied to every measurement in this queue; override per measurement if needed.
# Any parameter left out here keeps the measurement class's own default.
OVA_SETTINGS = {
    "center wavelength": 1566.72,                           # nm, C band
    # nm, must be one of the OVA's fixed spans:
    # 0.63, 1.27, 2.54, 5.09, 10.22, 20.58, 41.72, 85.78
    "wavelength range": "85.78",                            # widest span
    "Measurement Type": "Transmission",
    "Plot Measurement Type": "MIN_MAX_LOSS",
    "DUT L": 0.0,                                           # m
    "enable averaging": True,
    "number of averages": 10,
    "save_all_data": False,
    "filepath": "C:\\Users\\Luna\\Documents\\test.txt",
}

In [3]:
# --- Queue helpers ------------------------------------------------------------------
def switch_ports(output_channel):
    """M-port -> N-port routing that reads `output_channel` on the Luna OVA.

    The OVA source always drives the chip input channel; the OVA receiver follows the
    device's output channel. M1/M2 have no instrument attached, but the matrix switch
    cannot place two M ports on the same N port, so they are parked on unused channels.
    """
    if output_channel not in FIBRE_ARRAY_CHANNELS:
        raise ValueError(
            f"channel {output_channel} is not a fibre array channel {FIBRE_ARRAY_CHANNELS}"
        )
    if output_channel == INPUT_CHANNEL:
        raise ValueError(
            f"channel {output_channel} is the input channel, it cannot also be an output"
        )

    ports = {OVA_SOURCE_M_PORT: INPUT_CHANNEL, OVA_RECEIVE_M_PORT: output_channel}
    spare = [c for c in FIBRE_ARRAY_CHANNELS if c not in ports.values()]
    for m_port in (1, 2, 3, 4):
        if m_port not in ports:
            ports[m_port] = spare.pop(0)
    return {f"Switch Port: M = {m}": n for m, n in sorted(ports.items())}


def move(device_id):
    """Move the stages to a device."""
    return {"type": "move", "device_id": str(device_id)}


def sfp():
    """Run a Search for Peak at the current position."""
    return {"type": "sfp"}


def luna_meas(device_id, output_channel, **overrides):
    """One C-band Luna OVA measurement, with the switch oriented for `output_channel`."""
    parameters = dict(OVA_SETTINGS)
    parameters.update(switch_ports(output_channel))
    parameters.update(overrides)
    return {
        "type": "meas",
        "device_id": str(device_id),
        "measurement": MEASUREMENT_CLASS,
        "parameters": parameters,
    }

## The layout

One list per tile, giving the grating couplers **in physical left-to-right order**.
`INPUT` marks the coupler the fibre array input (channel 4) is aligned to; every other
entry is the device id read out on that coupler.

Channels are derived from each device's offset from `INPUT`, so if a tile is described in
the right order the switch routing follows automatically.

In [4]:
INPUT = None   # marks the fibre array input coupler within a tile

# Each tile: grating couplers left-to-right. Comments give the layout label of each GC.
TILES = [
    # tile 1 - input x = 0.0 um
    #   Loopback   Input   L2chain  L8chain  L6chain
    ["4310",       INPUT,  "4520",  "4680",  "4760"],

    # tile 2 - input x = 762.0 um
    #   L4chain  Loopback  Input   L10chain  L10chain
    ["4240",     "4311",   INPUT,  "4500",   "4601"],

    # tile 3 - input x = 1397.0 um
    #   L10chain  Loopback  Input   Loopback  L4chain
    ["4202",      "4312",   INPUT,  "4514",   "4641"],

    # tile 4 - input x = 2159.0 um
    #   L6chain  L8chain  L2chain  Input   Loopback
    ["4161",     "4281",  "4321",  INPUT,  "4517"],

    # tile 5 - input x = 2667.0 um
    #   L10chain  Loopback  Input   Loopback  L4chain
    ["4203",      "4313",   INPUT,  "4515",   "4642"],

    # tile 6 - input x = 3429.0 um
    #   L6chain  L8chain  L2chain  Input   Loopback
    ["4162",     "4282",  "4322",  INPUT,  "4516"],
]

In [5]:
# --- Build the queue ----------------------------------------------------------------
# Every device in a tile shares the tile's input coupler, so LabExT aligns once per tile
# and then measures all four devices at that one alignment. The stages are moved to the
# first device of the tile - any device of the tile would do, they share an input position.
entries = []
plan = []   # (tile number, device id, channel), kept for the preview and the cross-check

for tile_number, tile in enumerate(TILES, start=1):
    if tile.count(INPUT) != 1:
        raise ValueError(f"tile {tile_number} must mark exactly one INPUT coupler")
    input_index = tile.index(INPUT)

    tile_devices = []
    for gc_index, device_id in enumerate(tile):
        if device_id is INPUT:
            continue
        channel = INPUT_CHANNEL + (gc_index - input_index)
        if channel not in FIBRE_ARRAY_CHANNELS:
            raise ValueError(
                f"tile {tile_number}: device {device_id} maps to channel {channel}, "
                f"which is outside the fibre array {FIBRE_ARRAY_CHANNELS}"
            )
        tile_devices.append((device_id, channel))

    entries.append(move(tile_devices[0][0]))
    entries.append(sfp())
    for device_id, channel in tile_devices:
        entries.append(luna_meas(device_id, channel))
        plan.append((tile_number, device_id, channel))

print(f"{len(TILES)} tiles, {len(plan)} measurements, {len(entries)} queue entries")

6 tiles, 24 measurements, 36 queue entries


In [6]:
# --- Preview ------------------------------------------------------------------------
# Shows the queue the way LabExT will execute it. Measurements listed under one alignment
# step form a "block": LabExT requires every device in a block to sit at the same input
# location, since the stages are only aligned once for the whole block.
for i, entry in enumerate(entries):
    if entry["type"] == "move":
        print(f"{i:3d}  MOVE -> device {entry['device_id']}")
    elif entry["type"] == "sfp":
        print(f"{i:3d}  SEARCH FOR PEAK")
    else:
        p = entry["parameters"]
        routing = " ".join(f"M{m}=N{p[f'Switch Port: M = {m}']}" for m in (1, 2, 3, 4))
        print(f"{i:3d}      meas device {entry['device_id']:>6}  [{routing}]")

  0  MOVE -> device 4310
  1  SEARCH FOR PEAK
  2      meas device   4310  [M1=N1 M2=N2 M3=N4 M4=N3]
  3      meas device   4520  [M1=N1 M2=N2 M3=N4 M4=N5]
  4      meas device   4680  [M1=N1 M2=N2 M3=N4 M4=N6]
  5      meas device   4760  [M1=N1 M2=N2 M3=N4 M4=N7]
  6  MOVE -> device 4240
  7  SEARCH FOR PEAK
  8      meas device   4240  [M1=N1 M2=N3 M3=N4 M4=N2]
  9      meas device   4311  [M1=N1 M2=N2 M3=N4 M4=N3]
 10      meas device   4500  [M1=N1 M2=N2 M3=N4 M4=N5]
 11      meas device   4601  [M1=N1 M2=N2 M3=N4 M4=N6]
 12  MOVE -> device 4202
 13  SEARCH FOR PEAK
 14      meas device   4202  [M1=N1 M2=N3 M3=N4 M4=N2]
 15      meas device   4312  [M1=N1 M2=N2 M3=N4 M4=N3]
 16      meas device   4514  [M1=N1 M2=N2 M3=N4 M4=N5]
 17      meas device   4641  [M1=N1 M2=N2 M3=N4 M4=N6]
 18  MOVE -> device 4161
 19  SEARCH FOR PEAK
 20      meas device   4161  [M1=N2 M2=N3 M3=N4 M4=N1]
 21      meas device   4281  [M1=N1 M2=N3 M3=N4 M4=N2]
 22      meas device   4321  [M1=N1 M2=N2 M3=N

## Cross-check against the chip file (optional)

Confirms that every device named above exists on the chip and that the devices grouped
into one tile really do share an input coordinate - which is exactly what LabExT checks
when the queue is loaded. Skipped if the chip file is not next to this notebook.

In [7]:
import json
import os

CHIP_FILE = "test_tile_markers.json"

if not os.path.isfile(CHIP_FILE):
    print(f"{CHIP_FILE} not found next to this notebook - skipping cross-check.")
else:
    with open(CHIP_FILE) as f:
        chip_devices = {str(d["ID"]): d for d in json.load(f)}

    missing = sorted({device_id for _, device_id, _ in plan} - set(chip_devices))
    if missing:
        raise AssertionError(f"device ids not found in {CHIP_FILE}: {missing}")

    for tile_number, tile in enumerate(TILES, start=1):
        ids = [d for d in tile if d is not INPUT]
        positions = {tuple(chip_devices[d]["Inputs"][0]) for d in ids}
        if len(positions) != 1:
            raise AssertionError(
                f"tile {tile_number} devices do not share an input position: "
                + ", ".join(f"{d}={chip_devices[d]['Inputs'][0]}" for d in ids)
            )
        (position,) = positions
        types = ", ".join(f"{d} ({chip_devices[d]['Type']}) ch{c}"
                          for t, d, c in plan if t == tile_number)
        print(f"tile {tile_number}: input {list(position)} -> {types}")

    print("\nCross-check passed: all tiles are internally consistent.")

test_tile_markers.json not found next to this notebook - skipping cross-check.


In [8]:
# --- Write the queue file -----------------------------------------------------------
import json

queue = {"labext_queue_version": 1, "chip_name": CHIP_NAME, "entries": entries}

out_path = "test_tile_queue.json"
with open(out_path, "w") as f:
    json.dump(queue, f, indent=2)

print(f"Wrote {len(entries)} entries to {out_path}")
print("Load it in LabExT via File -> Load Experiment Queue...")

Wrote 36 entries to test_tile_queue.json
Load it in LabExT via File -> Load Experiment Queue...
